### **Modelos omni y evaluación multimodal temporal**

#### **Ablaciones, contradicciones, abstención, grounding y streaming**

Este cuaderno define un protocolo de evaluación para sistemas que reciben texto, audio y video. El objetivo es evitar una conclusión frecuente pero incorrecta: aceptar varias modalidades no demuestra que el sistema las integre.

La pregunta central es:

> ¿Qué evidencia experimental permite afirmar que un modelo omni utiliza, integra y sincroniza sus modalidades?


### **Preguntas de investigación**

#### **Hipótesis de trabajo**

**H1.** Aceptar varias modalidades no garantiza integración real.

**H2.** La ganancia multimodal debe compararse contra la mejor modalidad individual.

**H3.** Una política de abstención reduce errores ante contradicciones, pero también reduce cobertura.

**H4.** La respuesta puede ser correcta aunque el grounding temporal sea incorrecto.

**H5.** En streaming, una predicción final correcta puede ocultar inestabilidad durante la secuencia.


### **Configuración reproducible**

#### **Importaciones, rutas y semilla**


In [ ]:
from __future__ import annotations

import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def set_seed(seed: int) -> int:
    """Fija las semillas utilizadas en el cuaderno."""
    random.seed(seed)
    np.random.seed(seed)
    print("Semilla fijada:", seed)
    return seed

SEED = set_seed(227)
RESULTS_DIR = Path("resultados/cuaderno27_mcc225")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Directorio de resultados:", RESULTS_DIR)


### **Metadatos del experimento**

#### **Registro del protocolo**


In [ ]:
@dataclass
class ExperimentMetadata:
    course: str
    week: str
    notebook: str
    topic: str
    seed: int
    execution_mode: str


metadata = ExperimentMetadata(
    course="MCC225",
    week="Semana 12",
    notebook="Cuaderno27-MCC225",
    topic="Modelos omni y evaluación multimodal temporal",
    seed=SEED,
    execution_mode="CPU con protocolo simulado",
)

with open(RESULTS_DIR / "metadatos.json", "w", encoding="utf-8") as file:
    json.dump(asdict(metadata), file, indent=2, ensure_ascii=False)

asdict(metadata)


### **Definición operacional de modelo omni**

#### **Capacidades mínimas que deben documentarse**

Un sistema omni debe describirse mediante:

1. Modalidades de entrada.
2. Modalidades de salida.
3. Representación compartida o módulos especializados.
4. Tratamiento del tiempo físico.
5. Capacidad de streaming.
6. Manejo de modalidades faltantes.
7. Manejo de contradicciones.
8. Grounding de la evidencia.
9. Latencia y costo.
10. Controles de abstención.

El término omni no debe utilizarse solamente porque una interfaz acepte varios tipos de archivo.


### **Matriz de capacidades**

#### **Entradas, salidas y propiedades evaluadas**


In [ ]:
def build_capability_matrix() -> pd.DataFrame:
    """Construye una matriz mínima de capacidades omni."""
    rows = [
        {
            "entrada": "texto",
            "salida": "texto",
            "capacidad": "comprensión lingüística",
            "metrica_principal": "exactitud",
        },
        {
            "entrada": "audio",
            "salida": "texto",
            "capacidad": "comprensión auditiva",
            "metrica_principal": "exactitud por evento",
        },
        {
            "entrada": "video",
            "salida": "texto",
            "capacidad": "comprensión visual temporal",
            "metrica_principal": "exactitud y grounding",
        },
        {
            "entrada": "audio y video",
            "salida": "texto",
            "capacidad": "integración audiovisual",
            "metrica_principal": "ganancia multimodal",
        },
        {
            "entrada": "texto, audio y video",
            "salida": "texto",
            "capacidad": "integración omni",
            "metrica_principal": "exactitud, cobertura y conflicto",
        },
        {
            "entrada": "flujo audiovisual",
            "salida": "texto parcial",
            "capacidad": "procesamiento en streaming",
            "metrica_principal": "latencia y estabilidad",
        },
    ]
    return pd.DataFrame(rows)


capability_table = build_capability_matrix()
capability_table


### **Conjunto de casos omni**

#### **Evidencia complementaria, faltante y contradictoria**

Cada caso incluye probabilidades modales para una tarea binaria. Algunos casos son fáciles de resolver con una sola modalidad. Otros requieren combinar evidencia. También se incluyen modalidades faltantes y contradicciones deliberadas.


In [ ]:
@dataclass
class OmniCase:
    case_id: str
    label: int
    text_probability: float | None
    audio_probability: float | None
    video_probability: float | None
    conflict_expected: bool
    reference_start: int
    reference_end: int


def generate_omni_cases(num_cases: int, seed: int) -> list[OmniCase]:
    """Genera casos con evidencia concordante, complementaria y contradictoria."""
    rng = np.random.default_rng(seed)
    cases = []

    for case_index in range(num_cases):
        label = int(rng.integers(0, 2))
        base_probability = 0.78 if label == 1 else 0.22
        probabilities = np.clip(
            rng.normal(base_probability, 0.18, size=3),
            0.02,
            0.98,
        ).tolist()

        conflict_expected = case_index % 5 == 0
        if conflict_expected:
            probabilities[1] = 1.0 - base_probability

        if case_index % 7 == 0:
            probabilities[0] = None
        if case_index % 9 == 0:
            probabilities[2] = None

        reference_start = int(rng.integers(2, 14))
        reference_end = reference_start + int(rng.integers(3, 7))

        cases.append(OmniCase(
            case_id=f"caso_{case_index:03d}",
            label=label,
            text_probability=probabilities[0],
            audio_probability=probabilities[1],
            video_probability=probabilities[2],
            conflict_expected=conflict_expected,
            reference_start=reference_start,
            reference_end=reference_end,
        ))

    return cases


cases = generate_omni_cases(num_cases=120, seed=SEED)
case_table = pd.DataFrame([asdict(case) for case in cases])
case_table.head()


### **Políticas de integración**

#### **Ablaciones y combinación de modalidades**

Se evalúan las siguientes variantes:

1. Solo texto.
2. Solo audio.
3. Solo video.
4. Texto y audio.
5. Texto y video.
6. Audio y video.
7. Texto, audio y video.
8. Integración completa con abstención ante conflicto.


In [ ]:
MODALITY_FIELDS = {
    "texto": "text_probability",
    "audio": "audio_probability",
    "video": "video_probability",
}


def available_probabilities(
    case: OmniCase,
    modalities: tuple[str, ...],
) -> list[float]:
    """Recupera probabilidades disponibles para una selección de modalidades."""
    values = []
    for modality in modalities:
        value = getattr(case, MODALITY_FIELDS[modality])
        if value is not None:
            values.append(float(value))
    return values


def detect_probability_conflict(
    probabilities: list[float],
    threshold: float,
) -> bool:
    """Detecta conflicto cuando las modalidades apoyan clases opuestas."""
    if len(probabilities) < 2:
        return False
    has_positive = any(value >= threshold for value in probabilities)
    has_negative = any(value <= 1.0 - threshold for value in probabilities)
    return has_positive and has_negative


def predict_case(
    case: OmniCase,
    modalities: tuple[str, ...],
    decision_threshold: float,
    conflict_threshold: float,
    abstain_on_conflict: bool,
) -> dict[str, object]:
    """Predice una clase mediante promedio de probabilidades disponibles."""
    probabilities = available_probabilities(case, modalities)
    if len(probabilities) == 0:
        return {
            "prediction": None,
            "confidence": None,
            "abstained": True,
            "conflict": False,
        }

    conflict = detect_probability_conflict(probabilities, conflict_threshold)
    if abstain_on_conflict and conflict:
        return {
            "prediction": None,
            "confidence": float(np.mean(probabilities)),
            "abstained": True,
            "conflict": True,
        }

    confidence = float(np.mean(probabilities))
    prediction = int(confidence >= decision_threshold)
    return {
        "prediction": prediction,
        "confidence": confidence,
        "abstained": False,
        "conflict": conflict,
    }


variants = {
    "solo_texto": (("texto",), False),
    "solo_audio": (("audio",), False),
    "solo_video": (("video",), False),
    "texto_audio": (("texto", "audio"), False),
    "texto_video": (("texto", "video"), False),
    "audio_video": (("audio", "video"), False),
    "completo": (("texto", "audio", "video"), False),
    "completo_con_abstencion": (("texto", "audio", "video"), True),
}


### **Métricas de utilidad, cobertura y calibración**

#### **La exactitud no debe ocultar abstenciones**

Se reportan:

1. Exactitud sobre respuestas emitidas.
2. Cobertura.
3. Exactitud global, donde una abstención cuenta como no resuelta.
4. Tasa de abstención.
5. Tasa de conflicto detectado.
6. Brier score para predicciones con confianza.


In [ ]:
def evaluate_variant(
    cases: list[OmniCase],
    modalities: tuple[str, ...],
    abstain_on_conflict: bool,
) -> tuple[dict[str, float], list[dict[str, object]]]:
    """Evalúa una variante de integración omni."""
    traces = []
    answered_correctly = []
    global_correctness = []
    brier_values = []

    for case in cases:
        result = predict_case(
            case,
            modalities=modalities,
            decision_threshold=0.5,
            conflict_threshold=0.70,
            abstain_on_conflict=abstain_on_conflict,
        )
        prediction = result["prediction"]
        abstained = bool(result["abstained"])
        correct = prediction == case.label if prediction is not None else False
        if prediction is not None:
            answered_correctly.append(float(correct))
            brier_values.append((float(result["confidence"]) - case.label) ** 2)
        global_correctness.append(float(correct))

        traces.append({
            "identificador": case.case_id,
            "etiqueta": case.label,
            "prediccion": prediction,
            "confianza": result["confidence"],
            "abstencion": abstained,
            "conflicto_detectado": result["conflict"],
            "correcto": correct,
        })

    coverage = 1.0 - float(np.mean([trace["abstencion"] for trace in traces]))
    metrics = {
        "exactitud_selectiva": float(np.mean(answered_correctly)) if answered_correctly else 0.0,
        "cobertura": coverage,
        "exactitud_global": float(np.mean(global_correctness)),
        "tasa_abstencion": 1.0 - coverage,
        "tasa_conflicto": float(np.mean([trace["conflicto_detectado"] for trace in traces])),
        "brier": float(np.mean(brier_values)) if brier_values else 1.0,
    }
    return metrics, traces


summary_rows = []
trace_rows = []
for variant_name, (modalities, abstain_on_conflict) in variants.items():
    metrics, traces = evaluate_variant(
        cases,
        modalities=modalities,
        abstain_on_conflict=abstain_on_conflict,
    )
    summary_rows.append({"variante": variant_name, **metrics})
    for trace in traces:
        trace_rows.append({"variante": variant_name, **trace})

summary_table = pd.DataFrame(summary_rows)
trace_table = pd.DataFrame(trace_rows)
summary_table


In [ ]:
plt.figure(figsize=(10, 4))
positions = np.arange(len(summary_table))
plt.bar(
    positions - 0.18,
    summary_table["exactitud_global"],
    width=0.36,
    label="Exactitud global",
)
plt.bar(
    positions + 0.18,
    summary_table["cobertura"],
    width=0.36,
    label="Cobertura",
)
plt.xticks(positions, summary_table["variante"], rotation=30)
plt.ylim(0.0, 1.05)
plt.title("Utilidad y cobertura por variante")
plt.legend()
plt.tight_layout()
plt.show()


### **Ganancia multimodal**

#### **Comparación con la mejor modalidad individual**

La ganancia se define como:

\[
G = M_{completo} - \max(M_{texto}, M_{audio}, M_{video}).
\]

Una ganancia positiva sugiere complementariedad. No demuestra por sí sola que todas las modalidades sean utilizadas en cada caso.


In [ ]:
accuracy_map = dict(zip(summary_table["variante"], summary_table["exactitud_global"]))
best_unimodal = max(
    accuracy_map["solo_texto"],
    accuracy_map["solo_audio"],
    accuracy_map["solo_video"],
)
multimodal_gain = accuracy_map["completo"] - best_unimodal

print("Mejor exactitud unimodal:", best_unimodal)
print("Exactitud multimodal completa:", accuracy_map["completo"])
print("Ganancia multimodal:", multimodal_gain)


### **Contradicciones y abstención**

#### **Intercambio entre utilidad y seguridad**

La abstención puede reducir errores en casos conflictivos, pero también rechazar casos que podrían resolverse. Deben reportarse utilidad, cobertura y errores por subgrupo.


In [ ]:
conflict_case_ids = {case.case_id for case in cases if case.conflict_expected}
conflict_analysis_rows = []

for variant_name in ["completo", "completo_con_abstencion"]:
    current = trace_table[
        (trace_table["variante"] == variant_name)
        & (trace_table["identificador"].isin(conflict_case_ids))
    ]
    conflict_analysis_rows.append({
        "variante": variant_name,
        "numero_casos": len(current),
        "exactitud_global": float(current["correcto"].mean()),
        "tasa_abstencion": float(current["abstencion"].mean()),
        "tasa_conflicto_detectado": float(current["conflicto_detectado"].mean()),
    })

conflict_analysis_table = pd.DataFrame(conflict_analysis_rows)
conflict_analysis_table


### **Grounding temporal separado de la respuesta**

#### **Respuesta correcta y evidencia incorrecta**

Se simulan intervalos predichos. Una parte de los casos conserva la etiqueta correcta, pero desplaza el intervalo. El objetivo es demostrar que exactitud y grounding deben evaluarse por separado.


In [ ]:
def temporal_iou(
    predicted_start: int,
    predicted_end: int,
    reference_start: int,
    reference_end: int,
) -> float:
    """Calcula Intersection over Union temporal."""
    intersection = max(
        0,
        min(predicted_end, reference_end) - max(predicted_start, reference_start),
    )
    predicted_length = max(0, predicted_end - predicted_start)
    reference_length = max(0, reference_end - reference_start)
    union = predicted_length + reference_length - intersection
    return 0.0 if union == 0 else float(intersection / union)


rng = np.random.default_rng(SEED)
grounding_rows = []
complete_traces = trace_table[trace_table["variante"] == "completo"]
trace_by_id = complete_traces.set_index("identificador")

for case in cases:
    shift = int(rng.choice([0, 0, 0, -3, 3, 6]))
    predicted_start = max(0, case.reference_start + shift)
    predicted_end = max(predicted_start + 1, case.reference_end + shift)
    answer_correct = bool(trace_by_id.loc[case.case_id, "correcto"])
    grounding_rows.append({
        "identificador": case.case_id,
        "respuesta_correcta": answer_correct,
        "inicio_real": case.reference_start,
        "final_real": case.reference_end,
        "inicio_estimado": predicted_start,
        "final_estimado": predicted_end,
        "tiou": temporal_iou(
            predicted_start,
            predicted_end,
            case.reference_start,
            case.reference_end,
        ),
    })

grounding_table = pd.DataFrame(grounding_rows)
grounding_summary = grounding_table.groupby("respuesta_correcta")["tiou"].agg(
    ["count", "mean", "median"]
)
grounding_summary


In [ ]:
correct_with_bad_grounding = grounding_table[
    (grounding_table["respuesta_correcta"])
    & (grounding_table["tiou"] < 0.5)
]
print(
    "Respuestas correctas con grounding menor que 0.5:",
    len(correct_with_bad_grounding),
)
correct_with_bad_grounding.head()


### **Evaluación en streaming**

#### **Latencia, estabilidad y uso de evidencia futura**

En streaming, la evidencia llega por fragmentos. Se simula una secuencia de probabilidades acumuladas y se mide:

1. Tiempo hasta la primera predicción.
2. Tiempo hasta una predicción estable.
3. Número de cambios de decisión.
4. Confianza final.
5. Exactitud final.


In [ ]:
def simulate_streaming_probabilities(
    label: int,
    num_chunks: int,
    evidence_start: int,
    seed: int,
) -> np.ndarray:
    """Simula probabilidades parciales durante una entrada en streaming."""
    rng = np.random.default_rng(seed)
    probabilities = []
    current = 0.5

    for chunk_index in range(num_chunks):
        if chunk_index >= evidence_start:
            target = 0.85 if label == 1 else 0.15
            current = 0.65 * current + 0.35 * target
        current += float(rng.normal(0.0, 0.05))
        current = float(np.clip(current, 0.02, 0.98))
        probabilities.append(current)

    return np.asarray(probabilities)


def streaming_metrics(
    probabilities: np.ndarray,
    label: int,
    stability_window: int,
) -> dict[str, float]:
    """Calcula métricas de estabilidad para una trayectoria de streaming."""
    predictions = (probabilities >= 0.5).astype(int)
    flips = int(np.sum(predictions[1:] != predictions[:-1]))
    stable_index = len(predictions) - 1

    for start in range(0, len(predictions) - stability_window + 1):
        window = predictions[start:start + stability_window]
        if np.all(window == window[0]) and np.all(predictions[start:] == window[0]):
            stable_index = start
            break

    return {
        "primer_fragmento": 0.0,
        "fragmento_estable": float(stable_index),
        "cambios_decision": float(flips),
        "confianza_final": float(probabilities[-1]),
        "exactitud_final": float(predictions[-1] == label),
    }


streaming_rows = []
streaming_examples = {}
for case_index, case in enumerate(cases[:40]):
    probabilities = simulate_streaming_probabilities(
        label=case.label,
        num_chunks=18,
        evidence_start=5 + (case_index % 5),
        seed=2000 + case_index,
    )
    metrics = streaming_metrics(
        probabilities,
        label=case.label,
        stability_window=3,
    )
    streaming_rows.append({"identificador": case.case_id, **metrics})
    if case_index < 4:
        streaming_examples[case.case_id] = probabilities

streaming_table = pd.DataFrame(streaming_rows)
streaming_table.describe()


In [ ]:
plt.figure(figsize=(9, 4))
for case_id, probabilities in streaming_examples.items():
    plt.plot(probabilities, marker="o", label=case_id)
plt.axhline(0.5, linestyle="--", label="Umbral de decisión")
plt.xlabel("Fragmento recibido")
plt.ylabel("Probabilidad acumulada")
plt.title("Trayectorias de predicción en streaming")
plt.legend()
plt.tight_layout()
plt.show()


### **Incertidumbre mediante bootstrap**

#### **Intervalo para la ganancia multimodal**

La ganancia se estima por caso comparando corrección del sistema completo y de la mejor modalidad individual.


In [ ]:
def bootstrap_mean_interval(
    values: np.ndarray,
    repetitions: int,
    confidence_level: float,
    seed: int,
) -> tuple[float, float]:
    """Calcula un intervalo bootstrap para la media."""
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(repetitions):
        sample = rng.choice(values, size=len(values), replace=True)
        means.append(float(np.mean(sample)))
    alpha = 1.0 - confidence_level
    return (
        float(np.quantile(means, alpha / 2.0)),
        float(np.quantile(means, 1.0 - alpha / 2.0)),
    )


unimodal_variant = summary_table.sort_values(
    "exactitud_global",
    ascending=False,
).query("variante in ['solo_texto', 'solo_audio', 'solo_video']").iloc[0]["variante"]

complete_correct = trace_table[
    trace_table["variante"] == "completo"
].sort_values("identificador")["correcto"].astype(float).to_numpy()
unimodal_correct = trace_table[
    trace_table["variante"] == unimodal_variant
].sort_values("identificador")["correcto"].astype(float).to_numpy()
case_gains = complete_correct - unimodal_correct

gain_interval = bootstrap_mean_interval(
    case_gains,
    repetitions=2000,
    confidence_level=0.95,
    seed=SEED,
)
print("Mejor variante unimodal:", unimodal_variant)
print("Ganancia media por caso:", float(case_gains.mean()))
print("Intervalo bootstrap del 95 por ciento:", gain_interval)


### **Extensión opcional con un modelo omni abierto**

#### **Adaptador experimental**

La integración con un modelo real debe conservar el mismo conjunto de casos y registrar:

1. Modelo y revisión.
2. Modalidades realmente procesadas.
3. Frecuencia de audio.
4. Frames por segundo.
5. Duración y resolución.
6. Presupuesto de tokens.
7. Latencia al primer token.
8. Latencia al primer fragmento de audio.
9. Hardware.
10. Configuración de generación.


In [ ]:
RUN_REAL_MODEL = False

if RUN_REAL_MODEL:
    # Esta sección requiere dependencias, pesos y medios externos.
    # El adaptador debe devolver predicción, confianza, intervalo y latencia.
    print("Modo de modelo omni real activado.")
else:
    print("Modo de protocolo reproducible sin descargas externas.")


### **Amenazas a la validez**

#### **Límites de la evidencia experimental**

1. Las probabilidades son simuladas y no reproducen la correlación compleja entre modalidades reales.
2. El promedio de probabilidades es una política simple y no representa una arquitectura omni moderna.
3. El detector de contradicción utiliza umbrales fijos.
4. El streaming simula evidencia acumulada, pero no modela costos de codificación ni generación.
5. La ganancia multimodal puede depender de la construcción artificial de complementariedad.
6. La abstención debe analizarse junto con cobertura y no solamente con exactitud selectiva.


### **Conclusiones**

#### **Respuesta a la pregunta central**

Un sistema omni debe evaluarse mediante ablaciones, conflictos, modalidades faltantes, grounding y streaming. La exactitud completa no es suficiente. Se debe comparar con la mejor modalidad individual, medir cobertura, separar respuesta y evidencia, y cuantificar estabilidad temporal. El término omni debe sustentarse con un protocolo de capacidades y no solamente con una descripción comercial.


#### **Preguntas a desarrollar**

1. ¿Qué condiciones mínimas debe cumplir un sistema para denominarse omni?
2. ¿Por qué la mejor línea base unimodal es el comparador correcto?
3. ¿Cómo se interpreta una ganancia multimodal negativa?
4. ¿Qué relación existe entre abstención, cobertura y utilidad?
5. ¿Por qué una respuesta correcta puede tener grounding incorrecto?
6. ¿Cómo se mide estabilidad en streaming?
7. ¿Qué riesgos introduce el uso de información futura en una evaluación offline?
8. ¿Qué evidencia demostraría que el modelo utiliza el audio y no solamente el video?.


### **Referencias principales**

#### **Ruta de lectura**

1. Wu et al. **NExT-GPT: Any-to-Any Multimodal LLM**.
2. Zhan et al. **AnyGPT: Unified Multimodal LLM with Discrete Sequence Modeling**.
3. Défossez et al. **Moshi: A Speech-Text Foundation Model for Real-Time Dialogue**.
4. Xu et al. **Qwen2.5-Omni Technical Report**.
5. Fu et al. **Video-MME: Comprehensive Evaluation Benchmark of Multimodal LLMs in Video Analysis**.
6. Liu et al. **TempCompass: Do Video LLMs Really Understand Videos?**


### **Exportación de resultados**

#### **Evidencia reproducible**


In [ ]:
capability_table.to_csv(
    RESULTS_DIR / "matriz_capacidades.csv",
    index=False,
    encoding="utf-8",
)
case_table.to_csv(
    RESULTS_DIR / "casos_omni.csv",
    index=False,
    encoding="utf-8",
)
summary_table.to_csv(
    RESULTS_DIR / "ablaciones_modalidad.csv",
    index=False,
    encoding="utf-8",
)
trace_table.to_json(
    RESULTS_DIR / "trazas.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
conflict_analysis_table.to_csv(
    RESULTS_DIR / "analisis_conflictos.csv",
    index=False,
    encoding="utf-8",
)
grounding_table.to_csv(
    RESULTS_DIR / "grounding_temporal.csv",
    index=False,
    encoding="utf-8",
)
streaming_table.to_csv(
    RESULTS_DIR / "metricas_streaming.csv",
    index=False,
    encoding="utf-8",
)

summary = {
    "ganancia_multimodal": float(multimodal_gain),
    "intervalo_bootstrap_95": list(gain_interval),
    "mejor_variante_unimodal": str(unimodal_variant),
    "respuestas_correctas_grounding_bajo": int(len(correct_with_bad_grounding)),
}
with open(RESULTS_DIR / "resumen.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

print("Resultados exportados en:", RESULTS_DIR)
